# N22 — Record provisional screenshot QC
AI-assisted review; **not** independent human QC. Creates separate private CSV and summary; does not overwrite the original manual review template.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import csv, json

OUT = Path('/content/drive/MyDrive/thesis_all/outputs/runs/nb21_final_exact_n07_ensemble_fer')
INPUT = OUT / 'private_manual_face_qc.csv'
OUTPUT = OUT / 'private_ai_assisted_qc_provisional.csv'
SUMMARY = OUT / 'private_ai_assisted_qc_provisional_summary.json'

assert INPUT.is_file(), f'Missing file: {INPUT}'
with INPUT.open(newline='', encoding='utf-8') as f:
    original = list(csv.DictReader(f))
assert len(original) == 24
assert len({(r['source_id'], str(r['seed'])) for r in original}) == 24
# Pair indices follow notebook's sorted source_id,seed order.
original.sort(key=lambda r: (r['source_id'], int(r['seed'])))
flag_notes = {
    8: 'Possible distortion in eyes/eyelids; inspect full resolution',
    15: 'Possible localized mouth/eye artifacts; inspect full resolution',
    16: 'Possible localized mouth/eye artifacts; inspect full resolution',
    21: 'Strong facial changes and blur; inspect full resolution',
    22: 'Strong facial changes and blur; inspect full resolution',
}
result=[]
for i, record in enumerate(original, start=1):
    # Never override the authoritative human-review template.
    record = dict(record)
    status = 'UNCERTAIN' if i in flag_notes else 'PASS'
    for key in ('pairing','original_face','n07_face','n17_face','alignment'):
        record[key] = status if key in ('n07_face','n17_face') else 'PASS'
    record['notes'] = flag_notes.get(i, 'No obvious crop or severe facial distortion in screenshot; not full-resolution verified')
    record['reviewer'] = 'AI_ASSISTED_SCREENSHOT_PROVISIONAL_NOT_INDEPENDENT_HUMAN'
    record['pair_index'] = str(i)
    record['overall_status'] = status
    result.append(record)
assert sum(x['overall_status']=='PASS' for x in result)==19
assert sum(x['overall_status']=='UNCERTAIN' for x in result)==5
with OUTPUT.open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=list(result[0]))
    writer.writeheader();writer.writerows(result)
summary={'status':'PROVISIONAL_SCREENSHOT_REVIEW_NOT_FINAL_MANUAL_QC',
         'total':24,'pass':19,'uncertain':5,'fail':0,
         'uncertain_pairs':[8,15,16,21,22],
         'limitations':'Visual screenshot assessment only; original manual QC template untouched. Requires human confirmation/full-resolution check.'}
SUMMARY.write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('CREATED:',OUTPUT)
print('CREATED:',SUMMARY)
print('19 PASS / 5 UNCERTAIN / 0 FAIL')
print('Original private_manual_face_qc.csv unchanged.')
